# Setting up the notebook and realtive paths

In [1]:
# Discover repo root and read all CSV files from the per-series folders
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path (BEFORE the import attempt)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    print(f'Added {repo_root} to sys.path')

# Import the plot_head_distribution function
try:
    from functions.plot_functions import plot_head_distribution
    from functions.plot_functions import plot_groundwater_with_flags
except ImportError as e:
    print('✗ Failed to import plot_head_distribution:', e)
    print('  Verify that functions/plot_functions.py exists and contains the function.')
finally:
    print('Import attempt finished.')

functions_dir = repo_root / 'functions'
wiertsema_dir = repo_root / 'output_data' / 'csv_wiertsema_validated'
fugro_dir = repo_root / 'output_data' / 'csv_fugro_validated'
# Directory containing meteorological/stressor CSVs
stressor_dir = repo_root / 'input_stressors'
# Explicit stressor file paths used elsewhere in notebooks
precip_path = stressor_dir / 'knmi_berkhout_hourly_rain.csv'
evap_path = stressor_dir / 'knmi_berkhout_hourly_makkink.csv'

out_fig = repo_root / 'output_data' / 'figures'
out_fig.mkdir(parents=True, exist_ok=True)

print('wiertsema_dir ->', wiertsema_dir)
print('fugro_dir    ->', fugro_dir)
print('precip_path ->', precip_path)
print('evap_path  ->', evap_path)

Added d:\Users\jvanruitenbeek\data_validation to sys.path
Import attempt finished.
wiertsema_dir -> d:\Users\jvanruitenbeek\data_validation\output_data\csv_wiertsema_validated
fugro_dir    -> d:\Users\jvanruitenbeek\data_validation\output_data\csv_fugro_validated
precip_path -> d:\Users\jvanruitenbeek\data_validation\input_stressors\knmi_berkhout_hourly_rain.csv
evap_path  -> d:\Users\jvanruitenbeek\data_validation\input_stressors\knmi_berkhout_hourly_makkink.csv


# Creating the boxplots for a folder

In [2]:
# # Loop over all Fugro CSV files and generate head distribution plots

# # Create output directory for plots
# boxplot_output_dir = out_fig / 'wiertsema_data_distribution'
# boxplot_output_dir.mkdir(parents=True, exist_ok=True)

# # Get all CSV files from fugro folder
# input_boxplot_csv_folder = sorted(wiertsema_dir.glob('*.csv'))          #
# print(f'Found {len(input_boxplot_csv_folder)} CSV files\n')

# # Loop over each file
# for i, csv_file in enumerate(input_boxplot_csv_folder, start=1):
#     try:
#         print(f'[{i}/{len(input_boxplot_csv_folder)}] Processing: {csv_file.name}')
        
#         # Read the CSV
#         df = pd.read_csv(
#             csv_file,
#             index_col=0,
#             parse_dates=True,
#             encoding="utf-8-sig",
#             encoding_errors="replace"
#         )
        
#         # Coerce all columns to numeric
#         for col in df.columns:
#             df[col] = pd.to_numeric(df[col], errors="coerce")
        
#         # Select the first numeric column as the head data
#         numeric_cols = df.select_dtypes(include=['number']).columns
#         if len(numeric_cols) == 0:
#             print(f'  ✗ No numeric columns found\n')
#             continue
        
#         # The function expects a DataFrame with a "head" column, so rename the column
#         head_df = df[[numeric_cols[0]]].rename(columns={numeric_cols[0]: 'head'})
        
#         # Generate plot using the imported function
#         fig = plot_head_distribution(head_df, title=f'Head Distribution - {csv_file.stem}')
        
#         # Save as HTML
#         output_file = boxplot_output_dir / f'{csv_file.stem}.html'
#         fig.write_html(str(output_file))
#         print(f'  ✓ Saved: {output_file.name}\n')
        
#     except Exception as e:
#         print(f'  ✗ Error processing {csv_file.name}: {e}\n')

# print(f'✓ All plots saved to {boxplot_output_dir}')

# Creating the marked head plots for a folder 

In [3]:
# Loop over all CSV files and generate flagged head time series plots

# Create output directory for plots
timeseries_output_dir = out_fig / 'fugro_outliers_marked'
timeseries_output_dir.mkdir(parents=True, exist_ok=True)

# Setting the folder 
input_timeseries_csv_folder = sorted(fugro_dir.glob('*.csv'))
print(f'Found {len(input_timeseries_csv_folder)} CSV files\n')

# Helper mapping for flexible column names
evap_aliases = ["Evapotranspiration", "evaporation", "ET", "Evapo"]
prec_aliases = ["Precipitation", "precipitation", "Rain", "P"]

for i, csv_file in enumerate(input_timeseries_csv_folder, start=1):
    print(f'[{i}/{len(input_timeseries_csv_folder)}] Processing: {csv_file.name}')
    
    try:
        df = pd.read_csv(
            csv_file,
            index_col=0,          # eerste kolom als index (Time)
            parse_dates=[0],      # parse die kolom als datetime
            date_format='mixed',  # <-- belangrijk ivm .000 en zonder .000
            encoding="utf-8-sig",
            encoding_errors="replace",
        ).reset_index()          # Time weer terug als gewone kolom (optioneel)

        # Convert numeric columns safely
        df[df.columns[1:]] = df[df.columns[1:]].apply(pd.to_numeric, errors='coerce')

        # Auto-detect evaporation + precipitation columns
        evap_col = next((c for c in evap_aliases if c in df.columns), None)
        prec_col = next((c for c in prec_aliases if c in df.columns), None)

        # Printing dataframe info
        print(df.info())

        # Generate plot
        fig = plot_groundwater_with_flags(
            df,
            evap_col=evap_col,
            prec_col=prec_col
        )

        # Save output
        output_file = timeseries_output_dir / f'{csv_file.stem}.html'
        fig.write_html(str(output_file))
        print(f'  ✓ Saved\n')

    except Exception as e:
        print(f'  ✗ Error: {e}\n')

print(f'✓ All time series plots saved → {timeseries_output_dir}')

Found 156 CSV files

[1/156] Processing: NL-2412417-HWM_B09-PB1_m_NAP_avg.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18062 entries, 0 to 18061
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                18062 non-null  datetime64[ns]
 1   head                5 non-null      float64       
 2   head_raw            5 non-null      float64       
 3   Precipitation       18062 non-null  float64       
 4   Evapotranspiration  18062 non-null  float64       
 5   recharge            18062 non-null  float64       
 6   v0                  18062 non-null  bool          
 7   v1                  0 non-null      float64       
 8   v2                  0 non-null      float64       
 9   v3                  0 non-null      float64       
 10  v4                  0 non-null      float64       
dtypes: bool(1), datetime64[ns](1), float64(9)
memory usage: 1.4 MB
None
  ✓ Saved

[

# Creating Validated CSV files